# Inspect extracted spots and Friedel pairs

This notebook inspects

`../../data_esrf/all_friedelpairs/Al_big_grains_logtif.h5`

without modifying it. The file stores each variable-sized spot patch and mask in a flattened buffer. For spot row $i$, `offset[i]` gives the start and `shape[i] = (height, width)` gives the reshape dimensions.

The questions here are:

1. Are the flattened patch and mask buffers internally consistent?
2. What do individual detected spots look like across their size distribution?
3. Does `friedel_partner_id` correctly connect corresponding spots?
4. Do large or low-NCC blobs look like merged/multi-peak spots?
5. Does the file already contain two separated intensity targets?

In [ ]:
from pathlib import Path

import h5py
import hdf5plugin  # noqa: F401 - registers ESRF HDF5 compression filters
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np


def find_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "train.py").exists() and (candidate / "extract_spots").exists():
            return candidate
    raise RuntimeError("Could not find the project directory containing train.py.")


PROJECT_DIR = find_project_dir()
H5_FILE = PROJECT_DIR.parent / "data_esrf" / "all_friedelpairs" / "Al_big_grains_logtif.h5"
FIGURE_DIR = PROJECT_DIR / "extract_spots" / "h5_inspection_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not H5_FILE.exists():
    raise FileNotFoundError(H5_FILE)

print(f"Project: {PROJECT_DIR}")
print(f"HDF5:   {H5_FILE}")
print(f"Figures: {FIGURE_DIR}")

## 1. File structure and extraction configuration

The structure cell deliberately reads only metadata. It does not load the roughly 40 million patch values into memory.

In [ ]:
def decode_scalar(value):
    if isinstance(value, bytes):
        return value.decode(errors="replace")
    return value


def print_h5_structure(name, obj):
    kind = "Group" if isinstance(obj, h5py.Group) else "Dataset"
    shape = getattr(obj, "shape", "")
    dtype = getattr(obj, "dtype", "")
    print(f"{name} | {kind} | shape={shape} | dtype={dtype}")


with h5py.File(H5_FILE, "r") as f:
    f.visititems(print_h5_structure)
    print("\nformat_version:", f["im_seg"].attrs.get("format_version", "<missing>"))
    print("\nExtraction configuration:")
    for key, dataset in sorted(f["im_seg/config"].items()):
        print(f"  {key:22s}: {decode_scalar(dataset[()])}")

## 2. Load the compact spot table and validate the flattened storage

`friedel_partner_id` is a **blob ID**, not an array-row index. The `BLOB_TO_ROW` lookup below is therefore required before indexing a partner.

In [ ]:
SPOT_GROUP = "im_seg/spots"

with h5py.File(H5_FILE, "r") as f:
    spots = f[SPOT_GROUP]
    BLOB_ID = spots["blob_id"][:]
    CENTROID = spots["centroid"][:]
    FRAME_IDX = spots["frame_idx"][:]
    FRIEDEL_PARTNER_ID = spots["friedel_partner_id"][:]
    NCC_SCORE = spots["ncc_score"][:]
    OFFSET = spots["offset"][:]
    SHAPE = spots["shape"][:]
    N_PATCH_VALUES = spots["patches"].size
    N_MASK_VALUES = spots["masks"].size

N_SPOTS = len(BLOB_ID)
AREA = SHAPE[:, 0].astype(np.int64) * SHAPE[:, 1].astype(np.int64)
BLOB_TO_ROW = {int(blob_id): row for row, blob_id in enumerate(BLOB_ID)}
PARTNER_ROW = np.array([BLOB_TO_ROW.get(int(blob_id), -1) for blob_id in FRIEDEL_PARTNER_ID])
VALID_PARTNER = PARTNER_ROW >= 0
PAIR_ROWS = np.array(
    sorted({tuple(sorted((row, int(PARTNER_ROW[row])))) for row in np.flatnonzero(VALID_PARTNER)}),
    dtype=np.int64,
)

print(f"spots:                     {N_SPOTS:,}")
print(f"unique Friedel pairs:      {len(PAIR_ROWS):,}")
print(f"patch values:              {N_PATCH_VALUES:,}")
print(f"mask values:               {N_MASK_VALUES:,}")
print(f"offsets are cumulative:    {np.array_equal(OFFSET[1:], OFFSET[:-1] + AREA[:-1])}")
print(f"last patch ends at buffer: {int(OFFSET[-1] + AREA[-1]) == N_PATCH_VALUES}")
print(f"all partner IDs resolve:   {bool(VALID_PARTNER.all())}")
print(f"partner relation symmetric:{bool(np.all(PARTNER_ROW[PARTNER_ROW] == np.arange(N_SPOTS)))}")

frame_delta = np.abs(FRAME_IDX - FRAME_IDX[PARTNER_ROW])
print(f"partner frame delta:       min={frame_delta.min()}, median={np.median(frame_delta):.0f}, max={frame_delta.max()}")
print(f"NCC score range:           {NCC_SCORE.min():.3f} .. {NCC_SCORE.max():.3f}")

for name, values in [("height", SHAPE[:, 0]), ("width", SHAPE[:, 1]), ("area", AREA), ("NCC", NCC_SCORE)]:
    q = np.percentile(values, [0, 1, 25, 50, 75, 99, 100])
    print(f"{name:8s} [min,1%,25%,50%,75%,99%,max] = {np.round(q, 3)}")

## 3. Reconstruct and display individual spots

The intensity panel shows every stored patch value. The overlay uses the binary segmentation mask in green. Values outside the mask are retained in the stored patch, so the notebook shows both rather than silently multiplying them together.

In [ ]:
def read_spot(row=None, blob_id=None):
    if blob_id is not None:
        try:
            row = BLOB_TO_ROW[int(blob_id)]
        except KeyError as exc:
            raise KeyError(f"Unknown blob_id={blob_id}") from exc
    if row is None:
        raise ValueError("Give row or blob_id")
    row = int(row)
    if not 0 <= row < N_SPOTS:
        raise IndexError(row)

    height, width = map(int, SHAPE[row])
    start = int(OFFSET[row])
    stop = start + height * width
    with h5py.File(H5_FILE, "r") as f:
        group = f[SPOT_GROUP]
        patch = group["patches"][start:stop].reshape(height, width)
        mask = group["masks"][start:stop].reshape(height, width).astype(bool)

    return {
        "row": row,
        "blob_id": int(BLOB_ID[row]),
        "frame": int(FRAME_IDX[row]),
        "partner_blob_id": int(FRIEDEL_PARTNER_ID[row]),
        "partner_row": int(PARTNER_ROW[row]),
        "centroid": CENTROID[row].copy(),
        "ncc": float(NCC_SCORE[row]),
        "patch": patch,
        "mask": mask,
    }


def positive_limits(image, lower=1, upper=99.8):
    values = np.asarray(image)
    values = values[np.isfinite(values) & (values > 0)]
    if values.size == 0:
        return 1e-6, 1.0
    lo, hi = np.percentile(values, [lower, upper])
    lo = max(float(lo), 1e-6)
    hi = max(float(hi), lo * 1.01)
    return lo, hi


def draw_spot(ax, spot, overlay_mask=False, title=None):
    patch, mask = spot["patch"], spot["mask"]
    lo, hi = positive_limits(patch)
    ax.imshow(patch, cmap="gray", norm=LogNorm(vmin=lo, vmax=hi), interpolation="nearest")
    if overlay_mask:
        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        overlay[..., 1] = 1.0
        overlay[..., 3] = 0.28 * mask
        ax.imshow(overlay, interpolation="nearest")
    ax.set_title(title or f"row {spot['row']} | blob {spot['blob_id']}", fontsize=9)
    ax.axis("off")


quantiles = [5, 25, 50, 75, 95, 100]
area_targets = np.percentile(AREA, quantiles)
example_rows = [int(np.argmin(np.abs(AREA - target))) for target in area_targets]

fig, axes = plt.subplots(2, len(example_rows), figsize=(3 * len(example_rows), 6), constrained_layout=True)
for col, (percentile, row) in enumerate(zip(quantiles, example_rows)):
    spot = read_spot(row=row)
    title = f"area p{percentile}: row {row}\n{spot['patch'].shape}, mask px={spot['mask'].sum()}"
    draw_spot(axes[0, col], spot, title=title)
    draw_spot(axes[1, col], spot, overlay_mask=True, title=f"mask overlay | NCC={spot['ncc']:.3f}")
fig.suptitle("Representative detected spots across the stored patch-area distribution", fontsize=14)
fig.savefig(FIGURE_DIR / "representative_spots.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. Inspect matched Friedel pairs

Each row below is one matched pair. The partners occur approximately 1800 frames apart. Different crop shapes are expected because the two detections are stored independently.

In [ ]:
pair_area = AREA[PAIR_ROWS].sum(axis=1)
pair_ncc = NCC_SCORE[PAIR_ROWS[:, 0]]
selection = [
    int(np.argmin(np.abs(pair_area - np.percentile(pair_area, 25)))),
    int(np.argmin(np.abs(pair_area - np.percentile(pair_area, 50)))),
    int(np.argmin(np.abs(pair_area - np.percentile(pair_area, 75)))),
    int(np.argmax(pair_area)),
]

fig, axes = plt.subplots(len(selection), 4, figsize=(13, 3.3 * len(selection)), constrained_layout=True)
for plot_row, pair_number in enumerate(selection):
    row_a, row_b = map(int, PAIR_ROWS[pair_number])
    spot_a, spot_b = read_spot(row=row_a), read_spot(row=row_b)
    delta = abs(spot_a["frame"] - spot_b["frame"])
    draw_spot(
        axes[plot_row, 0], spot_a,
        title=f"A row {row_a}, blob {spot_a['blob_id']}\nframe {spot_a['frame']}, shape {spot_a['patch'].shape}",
    )
    draw_spot(axes[plot_row, 1], spot_a, overlay_mask=True, title="A mask overlay")
    draw_spot(
        axes[plot_row, 2], spot_b,
        title=f"B row {row_b}, blob {spot_b['blob_id']}\nframe {spot_b['frame']}, shape {spot_b['patch'].shape}",
    )
    draw_spot(
        axes[plot_row, 3], spot_b, overlay_mask=True,
        title=f"B mask overlay | delta={delta}, NCC={spot_a['ncc']:.3f}",
    )
fig.suptitle("Representative Friedel-pair spot crops", fontsize=14)
fig.savefig(FIGURE_DIR / "friedel_pair_examples.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Inspect candidates most likely to hide merged spots

There is no stored two-component decomposition. Large patch area and low pair NCC are only **inspection heuristics**, not proof of overlap. These panels deliberately emphasize the largest blobs and the weakest accepted Friedel matches.

In [ ]:
largest_rows = np.argsort(AREA)[-4:][::-1]
lowest_ncc_rows = PAIR_ROWS[np.argsort(pair_ncc)[:4], 0]
suspicious_rows = list(dict.fromkeys([*map(int, largest_rows), *map(int, lowest_ncc_rows)]))

fig, axes = plt.subplots(len(suspicious_rows), 3, figsize=(10, 2.8 * len(suspicious_rows)), constrained_layout=True)
for plot_row, row in enumerate(suspicious_rows):
    spot = read_spot(row=row)
    patch, mask = spot["patch"], spot["mask"]
    draw_spot(
        axes[plot_row, 0], spot,
        title=f"row {row}, blob {spot['blob_id']} | {patch.shape}\narea={AREA[row]}, NCC={spot['ncc']:.3f}",
    )
    draw_spot(axes[plot_row, 1], spot, overlay_mask=True, title=f"mask pixels={int(mask.sum())}")

    masked_values = patch[mask & np.isfinite(patch)]
    threshold = np.percentile(masked_values, 90) if masked_values.size else np.inf
    bright = mask & (patch >= threshold)
    axes[plot_row, 2].imshow(bright, cmap="magma", interpolation="nearest")
    axes[plot_row, 2].set_title("brightest 10% inside mask\n(multi-peak visual heuristic)", fontsize=9)
    axes[plot_row, 2].axis("off")

fig.suptitle("Large and low-NCC candidates for manual merged-spot inspection", fontsize=14)
fig.savefig(FIGURE_DIR / "possible_merged_spots.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Interactive helpers

Use either an array row or a `blob_id`. These functions always resolve Friedel partners through `BLOB_TO_ROW`; they never treat a blob ID as a row index.

In [ ]:
def show_spot(row=None, blob_id=None):
    spot = read_spot(row=row, blob_id=blob_id)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
    draw_spot(axes[0], spot, title=f"row {spot['row']} | blob {spot['blob_id']} | frame {spot['frame']}")
    draw_spot(axes[1], spot, overlay_mask=True, title=f"mask overlay | NCC={spot['ncc']:.3f}")
    axes[2].imshow(spot["patch"] * spot["mask"], cmap="gray", interpolation="nearest")
    axes[2].set_title("mask-applied intensity")
    axes[2].axis("off")
    plt.show()
    return spot


def show_friedel_pair(row=None, blob_id=None):
    first = read_spot(row=row, blob_id=blob_id)
    second = read_spot(row=first["partner_row"])
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), constrained_layout=True)
    draw_spot(axes[0], first, title=f"A row {first['row']} | blob {first['blob_id']}")
    draw_spot(axes[1], first, overlay_mask=True, title=f"frame {first['frame']}")
    draw_spot(axes[2], second, title=f"B row {second['row']} | blob {second['blob_id']}")
    draw_spot(axes[3], second, overlay_mask=True, title=f"frame {second['frame']} | NCC={first['ncc']:.3f}")
    plt.show()
    return first, second


# Change this row or use blob_id=... to inspect another example.
show_friedel_pair(row=int(PAIR_ROWS[len(PAIR_ROWS) // 2, 0]));

## Interpretation for the separation project

- The file contains **9,888 detected blob patches** arranged into **4,944 symmetric Friedel pairs**.
- Each detected blob has one intensity patch and one binary mask. These are useful isolated-spot candidates and are already cropped out of the full detector frames.
- A Friedel partner is the corresponding reflection roughly 1800 frames away. It is not a second component of an overlap in the same image.
- The file does **not** contain two separated intensity ground truths for a merged patch. If a stored blob actually contains overlapping spots, its single mask and single patch do not say which intensity belongs to which component.
- For separation training, the likely route is therefore still: quality-filter suitable single-spot donors, then generate controlled overlaps with two known intensity targets. The large/low-NCC panels help decide which candidates should be excluded or handled separately.